In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

In [ ]:

# =========================================================================
# 1. CREACIÓN DE DATOS CON PANDAS
# =========================================================================
np.random.seed(101)
meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
ventas_2026 = [45000, 48000, 52000, 58000, 61000, 65000, 72000, 70000, 78000, 82000, 88000, 95000]
proyeccion = [46000, 49000, 54000, 56000, 60000, 64000, 68000, 71000, 75000, 80000, 85000, 90000]

df_linea = pd.DataFrame({'Mes': meses, 'Ventas': ventas_2026, 'Meta': proyeccion})

categorias = ['Madera de Pino', 'Herramientas', 'Herrajes', 'Barnices y Tintes', 'Servicios de Corte']
ingresos_cat = [350000, 180000, 120000, 90000, 64000]
df_cat = pd.DataFrame({'Categoría': categorias, 'Ingresos': ingresos_cat})

vendedores = ['Carlos Gómez', 'Ana Martínez', 'Luis Rodríguez', 'Sofía Pérez']
ventas_vend = [240000, 210000, 190000, 164000]
df_vend = pd.DataFrame({'Vendedor': vendedores, 'Ventas': ventas_vend})

# =========================================================================
# 2. GENERACIÓN DE GRÁFICOS (GUARDADOS EN EL DIRECTORIO ACTUAL)
# =========================================================================
sns.set_theme(style='whitegrid')
color_principal = '#8B4513'   # Marrón Madera Elegante
color_secundario = '#D2B48C'  # Bronceado / Tan
color_oscuro = '#2F4F4F'      # Pizarra Oscuro

# Gráfico 1: Tendencia de Ventas (Línea + Área)
fig, ax = plt.subplots(figsize=(7.2, 3.2))
ax.plot(df_linea['Mes'], df_linea['Ventas'], marker='o', color=color_principal, linewidth=2.5, label='Ventas Reales 2026')
ax.plot(df_linea['Mes'], df_linea['Meta'], linestyle='--', color=color_secundario, linewidth=2, label='Meta Comercial')
ax.fill_between(df_linea['Mes'], df_linea['Ventas'], alpha=0.1, color=color_principal)
ax.set_title('Evolución Mensual de Ventas vs Meta', fontsize=12, fontweight='bold', color=color_oscuro, pad=10)
ax.set_ylabel('Monto ($ USD)', fontsize=10, color=color_oscuro)
ax.legend(frameon=True, facecolor='white', edgecolor='none')
plt.tight_layout()
plt.savefig('graf_linea.png', dpi=300)
plt.close()

# Gráfico 2: Composición por Categorías (Dona)
fig, ax = plt.subplots(figsize=(4.5, 3.5))
colors_pie = [color_principal, '#A0522D', '#CD853F', color_secundario, '#DEB887']
wedges, texts, autotexts = ax.pie(df_cat['Ingresos'], labels=df_cat['Categoría'], autopct='%1.1f%%',
                                  startangle=90, colors=colors_pie, wedgeprops=dict(width=0.4, edgecolor='w'))
plt.setp(autotexts, size=9, weight="bold", color="white")
plt.setp(texts, size=9, color=color_oscuro)
ax.set_title('Participación por Categoría', fontsize=12, fontweight='bold', color=color_oscuro, pad=10)
plt.tight_layout()
plt.savefig('graf_dona.png', dpi=300)
plt.close()

# Gráfico 3: Rendimiento de Vendedores (Barras Horizontales)
fig, ax = plt.subplots(figsize=(5, 3))
sns.barplot(x='Ventas', hue='Vendedor', data=df_vend, palette='copper', ax=ax, legend=False)
ax.set_title('Top Vendedores del Año', fontsize=12, fontweight='bold', color=color_oscuro, pad=10)
ax.set_xlabel('Ventas Acumuladas ($ USD)', fontsize=10)
ax.set_ylabel('', fontsize=10)
plt.tight_layout()
plt.savefig('graf_barras.png', dpi=300)
plt.close()

# =========================================================================
# 3. MAQUETACIÓN E IMPRESIÓN DEL REPORTE PDF CON REPORTLAB
# =========================================================================
doc = SimpleDocTemplate(
    "reporte_ventas_la_casa_del_carpintero.pdf",
    pagesize=letter,
    rightMargin=36, leftMargin=36, topMargin=36, bottomMargin=36
)

styles = getSampleStyleSheet()

# Estilos tipográficos institucionales
style_titulo_principal = ParagraphStyle(
    'TituloPrincipal', parent=styles['Heading1'], fontName='Helvetica-Bold',
    fontSize=26, textColor=colors.HexColor(color_principal), spaceAfter=6, alignment=1
)
style_subtitulo = ParagraphStyle(
    'Subtitulo', parent=styles['Normal'], fontName='Helvetica',
    fontSize=12, textColor=colors.HexColor('#555555'), spaceAfter=20, alignment=1
)
style_seccion = ParagraphStyle(
    'Seccion', parent=styles['Heading2'], fontName='Helvetica-Bold',
    fontSize=16, textColor=colors.HexColor(color_oscuro), spaceBefore=12, spaceAfter=10
)
style_body = ParagraphStyle(
    'CuerpoTexto', parent=styles['BodyText'], fontName='Helvetica',
    fontSize=10, textColor=colors.HexColor('#333333'), spaceAfter=8
)
style_kpi_val = ParagraphStyle('KpiVal', fontName='Helvetica-Bold', fontSize=18, textColor=colors.HexColor(color_principal), alignment=1)
style_kpi_lbl = ParagraphStyle('KpiLbl', fontName='Helvetica', fontSize=9, textColor=colors.HexColor('#666666'), alignment=1)
style_th = ParagraphStyle('TableHead', fontName='Helvetica-Bold', fontSize=10, textColor=colors.white, alignment=1)
style_td = ParagraphStyle('TableData', fontName='Helvetica', fontSize=9, textColor=colors.HexColor('#333333'), alignment=1)

story = []

# --- PÁGINA 1: DASHBOARD PRINCIPAL ---
story.append(Paragraph("LA CASA DEL CARPINTERO", style_titulo_principal))
story.append(Paragraph("Informe Corporativo Anual de Desempeño de Ventas", style_subtitulo))
story.append(Spacer(1, 10))

# Cuadrícula de KPIs estilo Dashboard
kpi_data = [
    [
        Paragraph("<b>$804,000 USD</b>", style_kpi_val),
        Paragraph("<b>+14.2%</b>", style_kpi_val),
        Paragraph("<b>$67,000 USD</b>", style_kpi_val),
        Paragraph("<b>89.5%</b>", style_kpi_val)
    ],
    [
        Paragraph("Ingresos Totales Brutos", style_kpi_lbl),
        Paragraph("Crecimiento vs Año Ant.", style_kpi_lbl),
        Paragraph("Ticket Promedio Mensual", style_kpi_lbl),
        Paragraph("Eficiencia de Cumplimiento", style_kpi_lbl)
    ]
]
kpi_table = Table(kpi_data, colWidths=[135, 135, 135, 135])
kpi_table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#FDF5E6')),
    ('BOX', (0,0), (-1,-1), 1, colors.HexColor(color_secundario)),
    ('INNERGRID', (0,0), (-1,-1), 0.5, colors.HexColor('#E6C29E')),
    ('TOPPADDING', (0,0), (-1,-1), 12),
    ('BOTTOMPADDING', (0,0), (-1,-1), 12),
]))
story.append(kpi_table)
story.append(Spacer(1, 20))

story.append(Paragraph("1. Resumen Ejecutivo de Tendencias Financieras", style_seccion))
story.append(Paragraph("Durante el presente ejercicio fiscal, <b>La Casa Del Carpintero</b> ha consolidado su posición líder en el mercado de suministros madereros y herramientas especializadas. La gráfica inferior muestra el progreso ascendente consolidado mes a mes superando las proyecciones iniciales a partir del segundo semestre del año.", style_body))
story.append(Spacer(1, 10))

story.append(Image('graf_linea.png', width=7.2*inch, height=3.2*inch))
story.append(PageBreak())

# --- PÁGINA 2: UNIDADES DE NEGOCIO ---
story.append(Paragraph("2. Rendimiento por Unidades de Negocio", style_seccion))
story.append(Paragraph("El análisis de segmentación resalta que la <b>Madera de Pino</b> continúa representando el pilar fundamental de facturación de la organización, seguido de forma robusta por el catálogo de <b>Herramientas Mecánicas y Eléctricas</b>.", style_body))
story.append(Spacer(1, 15))

cat_table_data = [[Paragraph("<b>Categoría de Producto</b>", style_th), Paragraph("<b>Ventas Anuales ($ USD)</b>", style_th), Paragraph("<b>Participación %</b>", style_th)]]
for idx, row in df_cat.iterrows():
    porcentaje = (row['Ingresos'] / sum(ingresos_cat)) * 100
    cat_table_data.append([
        Paragraph(row['Categoría'], style_td),
        Paragraph(f"${row['Ingresos']:,} USD", style_td),
        Paragraph(f"{porcentaje:.1f}%", style_td)
    ])

cat_table = Table(cat_table_data, colWidths=[200, 170, 170])
cat_table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor(color_principal)),
    ('BOTTOMPADDING', (0,0), (-1,-1), 8),
    ('TOPPADDING', (0,0), (-1,-1), 8),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor(color_secundario)),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#F9F6F0')])
]))
story.append(cat_table)
story.append(Spacer(1, 20))
story.append(Image('graf_dona.png', width=4.5*inch, height=3.5*inch))
story.append(PageBreak())

# --- PÁGINA 3: FUERZA DE VENTAS Y DATOS DETALLADOS ---
story.append(Paragraph("3. Operaciones Comerciales y Fuerza de Ventas", style_seccion))
story.append(Paragraph("A continuación, se detalla el desglose completo del histórico de facturación junto con las métricas individuales de nuestro equipo comercial, quienes lograron cuotas sobresalientes este periodo.", style_body))
story.append(Spacer(1, 10))

story.append(Image('graf_barras.png', width=5*inch, height=3*inch))
story.append(Spacer(1, 15))

mes_table_data = [[Paragraph("<b>Mes</b>", style_th), Paragraph("<b>Ventas Logradas</b>", style_th), Paragraph("<b>Meta Estipulada</b>", style_th), Paragraph("<b>Desviación</b>", style_th)]]
for idx, row in df_linea.iterrows():
    desv = row['Ventas'] - row['Meta']
    desv_str = f"+${desv:,}" if desv >= 0 else f"-${abs(desv):,}"
    mes_table_data.append([
        Paragraph(row['Mes'], style_td),
        Paragraph(f"${row['Ventas']:,}", style_td),
        Paragraph(f"${row['Meta']:,}", style_td),
        Paragraph(desv_str, style_td)
    ])

mes_table = Table(mes_table_data, colWidths=[100, 150, 150, 140])
mes_table.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor(color_oscuro)),
    ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ('TOPPADDING', (0,0), (-1,-1), 5),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#CCCCCC')),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#CCCCCC')),
    ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.white, colors.HexColor('#F4F4F4')])
    ]))
story.append(mes_table)
doc.build(story)

print("¡Reporte PDF generado exitosamente!")
